In [64]:
import pandas as pd

dataset link :- "https://www.kaggle.com/datasets/kukuroo3/churn-model-data-set-competition-form/data?select=y_train.csv"

In [65]:
data = pd.read_csv('X_test.csv')
data = data.drop(columns=["CustomerId"])
display(data.head())
print(data.shape)

,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,Abdullah,802,France,Female,60,3,92887.06,1,1,0,39473.63
1,Ignatiev,602,France,Female,56,3,115895.22,3,1,0,4176.17
2,Anenechukwu,801,France,Female,32,4,75170.54,1,1,1,37898.50
3,Wade,693,Spain,Female,34,10,107556.06,2,0,0,154631.35
4,Ch'in,592,France,Female,62,5,0.00,1,1,1,100941.57


(3501, 11)


In [66]:
data["Geography"].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

# Pre-Process data

In [67]:
df = data.drop(columns=["Surname"])
df["Gender"] = df["Gender"].map({"Female": 0, "Male": 1})
df["Geography"] = df["Geography"].astype("category").cat.codes
x = df
x.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,802,0,0.0,60,3,92887.06,1,1,0,39473.63
1,602,0,0.0,56,3,115895.22,3,1,0,4176.17
2,801,0,0.0,32,4,75170.54,1,1,1,37898.50
3,693,2,0.0,34,10,107556.06,2,0,0,154631.35
4,592,0,0.0,62,5,0.00,1,1,1,100941.57


In [68]:
x["Geography"].unique()

array([0, 2, 1], dtype=int8)

In [69]:
display(x.info())
print("Size of data is ",end=" ");display(x.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3501 entries, 0 to 3500
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      3501 non-null   int64  
 1   Geography        3501 non-null   int8   
 2   Gender           3401 non-null   float64
 3   Age              3501 non-null   int64  
 4   Tenure           3501 non-null   int64  
 5   Balance          3501 non-null   float64
 6   NumOfProducts    3501 non-null   int64  
 7   HasCrCard        3501 non-null   int64  
 8   IsActiveMember   3501 non-null   int64  
 9   EstimatedSalary  3501 non-null   float64
dtypes: float64(3), int64(6), int8(1)
memory usage: 249.7 KB


None

Size of data is  

(3501, 10)

 ## -- > 2   Gender           3401 non-null   float64  <-- Missing value are there


In [70]:
print(x.isna().sum())

CreditScore          0
Geography            0
Gender             100
Age                  0
Tenure               0
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
dtype: int64


In [71]:
x["Gender"] = x["Gender"].fillna(x["Gender"].median())
print("Missing values in Gender:", x["Gender"].isna().sum())

Missing values in Gender: 0


In [72]:
x["Balance"] = x["Balance"] / 100000.0
x["EstimatedSalary"] = x["EstimatedSalary"] / 50000.0
x['CreditScore'] = x['CreditScore'] / 650
x.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,1.233846,0,0.0,60,3,0.928871,1,1,0,0.789473
1,0.926154,0,0.0,56,3,1.158952,3,1,0,0.083523
2,1.232308,0,0.0,32,4,0.751705,1,1,1,0.757970
3,1.066154,2,0.0,34,10,1.075561,2,0,0,3.092627
4,0.910769,0,0.0,62,5,0.000000,1,1,1,2.018831


In [73]:
trg = pd.read_csv('y_test.csv')
trg = trg.drop(columns=["CustomerId"])
# trg = trg["Exited"].unique()# - > [1 0]
trg.head(5)

,Exited
0,1
1,1
2,0
3,0
4,0


# Helper Functions


In [74]:
import jax
import jax.numpy as jnp
from sklearn.preprocessing import StandardScaler
import numpy as np
from typing import Callable, List

# -----------------------------
# Activation functions
# -----------------------------
def relu(x):
    return jnp.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + jnp.exp(-x))

# -----------------------------
# Utility functions
# -----------------------------
def _get_preferred_device(prefer_gpu: bool = True):
    """Return the preferred JAX device (GPU if available, else CPU)."""
    if prefer_gpu:
        gpus = jax.devices("gpu")
        if gpus:
            return gpus[0]
    return jax.devices()[0]

def get_row(df: pd.DataFrame, index: int):
    """
    Return a single row as an (n x 1) NumPy matrix (column vector).
    """
    row = df.iloc[index].to_numpy().reshape(-1, 1)
    return row

def create_random_matrix(x: int, y: int, low: float = 0.0, high: float = 2.0,
                         dtype=jnp.float32, key=None, device=None):
    """Create x-by-y JAX array with uniform random floats"""
    if key is None:
        key = jax.random.PRNGKey(0)
    if device is None:
        device = _get_preferred_device(prefer_gpu=True)
    arr = jax.random.uniform(key, (x, y), minval=low, maxval=high, dtype=dtype)
    arr = jax.device_put(arr, device=device)
    return arr

def forward_pass(weight, X, bias):
    """Forward pass: W^T * X + b"""
    return jnp.dot(weight, X) + bias

def Loss_funtion(ypred, yorg):
    """MSE Loss using jax.numpy"""
    return jnp.mean((ypred - yorg) ** 2)

# -----------------------------
# Backpropagation
# -----------------------------
def model_loss(weight1, bias1, weight2, bias2, X, y_true, act_fun):
    hidden = forward_pass(weight1, X, bias1)
    hidden = act_fun(hidden)
    y_pred = forward_pass(weight2, hidden, bias2)
    return Loss_funtion(y_pred, y_true)

def backprop(weight1, bias1, weight2, bias2, X, y_true, act_fun):
    grad_fn = jax.grad(model_loss, argnums=(0,1,2,3))
    dW1, dB1, dW2, dB2 = grad_fn(weight1, bias1, weight2, bias2, X, y_true, act_fun)
    return dW1, dB1, dW2, dB2

def update_WB(dW1, dB1, dW2, dB2, lr=0.001):
    """Update global weight/bias variables in-place"""
    global weight, bais, weight2, bais2
    weight  -= lr * dW1
    bais    -= lr * dB1
    weight2 -= lr * dW2
    bais2   -= lr * dB2

def _get_preferred_device(prefer_gpu: bool = True):
    """Return the preferred JAX device (GPU if available, else CPU)."""
    if prefer_gpu:
        gpus = jax.devices("gpu")
        if gpus:
            return gpus[0]
    return jax.devices()[0]

def create_random_matrix(x: int, y: int, low: float = 0.0, high: float = 2.0,
                         dtype=np.float32, key=None, device=None):
    """
    Create an x-by-y JAX array of random floats uniformly sampled in [low, high].

    Args:
      x, y: matrix dimensions
      low, high: range for random values
      dtype: jnp dtype (default: float32)
      key: optional jax.random.PRNGKey
      device: optional jax device (defaults to GPU if available)

    Returns:
      JAX array on chosen device
    """
    if key is None:
        key = jax.random.PRNGKey(0)  # deterministic seed if none given
    if device is None:
        device = _get_preferred_device(prefer_gpu=True)

    # uniform random floats in [low, high)
    arr = jax.random.uniform(key, (x, y), minval=low, maxval=high, dtype=dtype)
    arr = jax.device_put(arr, device=device)
    return arr


## NN Dsin 12 → 16 → 1

In [75]:
def Personal_NN(X: jnp.ndarray, y_org: jnp.ndarray, lr: float = 0.001,
                epochs: int = 10, act_fun: Callable = relu,
                num_epo_print: int = 1) -> List[jnp.ndarray]:
    """
    Simple 2-layer NN with customizable activation function for hidden layer.

    Args:
        X: Input column vector (n x 1)
        y_org: Target scalar (standardized)
        lr: Learning rate
        epochs: Number of training epochs
        act_fun: Activation function for hidden layer (callable)
        num_epo_print: Print loss every `num_epo_print` epochs

    Returns:
        weight, bais, weight2, bais2
    """
    global weight, bais, weight2, bais2

    # y_org = y_org.reshape(1, -1)

    # --- UPDATED WEIGHT DIMENSIONS for 12 → 16 → 1 ---
    # Layer 1: Input (12) → Hidden (16)
    weight  = create_random_matrix(8, 10, key=jax.random.PRNGKey(42))
    bais    = create_random_matrix(8, 1,  key=jax.random.PRNGKey(41))

    # Layer 2: Hidden (16) → Output (1)
    weight2 = create_random_matrix(1, 8, key=jax.random.PRNGKey(40))
    bais2   = create_random_matrix(1, 1,  key=jax.random.PRNGKey(39))

    # --- Initial forward pass ---
    hidden = forward_pass(weight, X, bais)
    hidden = act_fun(hidden)
    y_pred = forward_pass(weight2, hidden, bais2)

    if num_epo_print > 0:
        print("Initial Training loss:", np.array(Loss_funtion(y_pred, y_org)))

    # --- Training loop ---
    for epoch in range(epochs):
        dW1, dB1, dW2, dB2 = backprop(weight, bais, weight2, bais2, X, y_org, act_fun)
        update_WB(dW1, dB1, dW2, dB2, lr)

        if num_epo_print > 0 and ((epoch + 1) % num_epo_print == 0 or epoch == epochs - 1):
            hidden = forward_pass(weight, X, bais)
            hidden = act_fun(hidden)
            y_pred = forward_pass(weight2, hidden, bais2)
            loss = Loss_funtion(y_pred, y_org)
            print(f"Epoch {epoch + 1} Training loss:", np.array(loss))

    return weight, bais, weight2, bais2

def predict(X, weight1, bias1, weight2, bias2, act_fun: Callable = relu,
            scaler_y=None):
    hidden = forward_pass(weight1, X, bias1)
    hidden = act_fun(hidden)
    y_pred_std = forward_pass(weight2, hidden, bias2)

    if scaler_y is not None:
        # reshape to 2D for StandardScaler
        y_pred_std_np = np.array(y_pred_std).reshape(-1, 1)
        y_pred_original = scaler_y.inverse_transform(y_pred_std_np)
        return y_pred_original[0,0]   # return scalar
    else:
        return y_pred_std

# Test case

In [76]:
trg.head()

,Exited
0,1
1,1
2,0
3,0
4,0


In [77]:
# Number of inputs are 10
X = np.array([[1.233846, 0, 0.0, 60, 3, 0.928871, 1, 1, 0, 0.789473]], dtype=jnp.float32)
y = np.array([[1]], dtype=jnp.float32)

# Standardize
scaler_X = StandardScaler()
X_std = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_std = scaler_y.fit_transform(y)

# Convert to JAX tensors (note .T to get (12, 1))
X_jax = jnp.array(X_std.T, dtype=jnp.float32)
y_jax = jnp.array(y_std[0, 0], dtype=jnp.float32)

# Train
weight, bais, weight2, bais2 = Personal_NN(
    X_jax, y_jax,
    lr=0.01,
    epochs=100,
    act_fun=relu,
    num_epo_print=10
)

# Predict
y_pred_original = predict(X_jax, weight, bais, weight2, bais2, act_fun=relu, scaler_y=scaler_y)
print("\nPredicted output (original scale):", y_pred_original)


Initial Training loss: 23.801083
Epoch 10 Training loss: 0.16024642
Epoch 20 Training loss: 0.0020887405
Epoch 30 Training loss: 2.7147178e-05
Epoch 40 Training loss: 3.5259413e-07
Epoch 50 Training loss: 4.5867807e-09
Epoch 60 Training loss: 5.820766e-11
Epoch 70 Training loss: 1.2159163e-12
Epoch 80 Training loss: 8.881784e-14
Epoch 90 Training loss: 5.684342e-14
Epoch 100 Training loss: 3.1974423e-14

Predicted output (original scale): 1.0000002


# Now Train on data

In [78]:
X_jax = np.array(x, dtype=jnp.float32)
y_jax = np.array(y, dtype=jnp.float32).reshape(1, -1)
display(X_jax.shape)
display(y_jax.shape)

(3501, 10)

(1, 1)

In [79]:
weight, bais, weight2, bais2 = Personal_NN(
    X_jax.T, y_jax,
    lr=0.01,
    epochs=1000,
    act_fun=relu,
    num_epo_print=100
)

Initial Training loss: 121575.98
Epoch 100 Training loss: 0.947833
Epoch 200 Training loss: 0.01667042
Epoch 300 Training loss: 0.0002932027
Epoch 400 Training loss: 5.1563425e-06
Epoch 500 Training loss: 9.067484e-08
Epoch 600 Training loss: 1.5900561e-09
Epoch 700 Training loss: 2.9420022e-11
Epoch 800 Training loss: 2.220446e-12
Epoch 900 Training loss: 2.220446e-12
Epoch 1000 Training loss: 2.220446e-12


Now let's test some sample

In [87]:
# Load test data
test_df = pd.read_csv('X_train.csv')  # or your test CSV file

# Drop non-feature columns
#

# Encode categorical variables
test_df["Gender"] = test_df["Gender"].map({"Female": 0, "Male": 1})
test_df["Geography"] = test_df["Geography"].astype("category").cat.codes
test_df = test_df.drop(columns=["CustomerId", "Surname"])
# Scale numeric features
test_df["Balance"] = test_df["Balance"] / 100000.0
test_df["EstimatedSalary"] = test_df["EstimatedSalary"] / 50000.0
test_df['CreditScore'] = test_df['CreditScore'] / 650
display(test_df.head())
# Convert to JAX array for Personal_NN

y_test = pd.read_csv('y_train.csv')
y_test = y_test.drop(columns=["CustomerId"])
y_test = np.array(y_test, dtype=jnp.float32)

X_test_jax = np.array(test_df.values, dtype=jnp.float32)
y_test_jax = np.array(y_test, dtype=jnp.float32).reshape(1, -1)
print(y_test_jax[0, :6])

print(X_test_jax.shape, y_test_jax.shape)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,1.216923,1,0.0,35,7,0.524362,1,1,0,3.221035
1,1.084615,1,1.0,42,8,1.666859,2,1,1,1.106270
2,0.835385,0,0.0,31,4,1.383179,1,0,0,1.236875
3,1.090769,0,0.0,32,2,0.000000,2,0,0,2.193626
4,1.098462,1,0.0,36,1,1.016090,2,1,1,0.008955


[0. 0. 0. 0. 0. 1.]
(6499, 10) (1, 6499)


In [89]:
#Make prediction
y_pred_test = predict(X_test_jax.T, weight, bais, weight2, bais2, act_fun=relu, scaler_y=scaler_y)
y_pred_class = (y_pred_test > 0.5).astype(int)

# Add predictions to dataframe
test_df["Predicted_Exited"] = y_pred_class

# Display first 5 predictions
display(test_df.head())




,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Predicted_Exited
0,1.216923,1,0.0,35,7,0.524362,1,1,0,3.221035,1
1,1.084615,1,1.0,42,8,1.666859,2,1,1,1.106270,1
2,0.835385,0,0.0,31,4,1.383179,1,0,0,1.236875,1
3,1.090769,0,0.0,32,2,0.000000,2,0,0,2.193626,1
4,1.098462,1,0.0,36,1,1.016090,2,1,1,0.008955,1


## Clean code

In [101]:
# -----------------------------
# 1️⃣ Import libraries
# -----------------------------
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
from typing import Callable, List

# -----------------------------
# 2️⃣ Activation functions
# -----------------------------
def relu(x):
    return jnp.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + jnp.exp(-x))

# -----------------------------
# 3️⃣ Utility functions
# -----------------------------
def forward_pass(weight, X, bias):
    """Forward pass: W @ X + b"""
    return jnp.dot(weight, X) + bias

def Loss_funtion(y_pred, y_true):
    """Mean Squared Error loss"""
    return jnp.mean((y_pred - y_true) ** 2)

def create_random_matrix(x: int, y: int, key=None):
    """Random weight initialization"""
    if key is None:
        key = jax.random.PRNGKey(0)
    return jax.random.uniform(key, (x, y), minval=-0.5, maxval=0.5, dtype=jnp.float32)

def backprop(weight1, bias1, weight2, bias2, X, y_true, act_fun):
    """Compute gradients"""
    def model_loss(w1, b1, w2, b2):
        hidden = forward_pass(w1, X, b1)
        hidden = act_fun(hidden)
        y_pred = forward_pass(w2, hidden, b2)
        return Loss_funtion(y_pred, y_true)
    grad_fn = jax.grad(model_loss, argnums=(0,1,2,3))
    return grad_fn(weight1, bias1, weight2, bias2)

def update_WB(dW1, dB1, dW2, dB2, lr=0.001):
    """Update global weight/bias variables"""
    global weight, bais, weight2, bais2
    weight  -= lr * dW1
    bais    -= lr * dB1
    weight2 -= lr * dW2
    bais2   -= lr * dB2

# -----------------------------
# 4️⃣ Neural Network
# -----------------------------
def Personal_NN(X: jnp.ndarray, y_org: jnp.ndarray, lr: float = 0.001,
                epochs: int = 100, act_fun: Callable = relu,
                num_epo_print: int = 10) -> List[jnp.ndarray]:
    """
    Simple 2-layer NN (10-8-1) for batch training
    """
    global weight, bais, weight2, bais2

    # Ensure y_org shape = (1, num_samples)
    y_org = y_org.reshape(1, -1)

    num_samples, num_features = X.shape
    hidden_dim = 8
    output_dim = 1

    # Initialize weights and biases
    weight = create_random_matrix(hidden_dim, num_features, key=jax.random.PRNGKey(42))
    bais   = jnp.zeros((hidden_dim, 1))
    weight2 = create_random_matrix(output_dim, hidden_dim, key=jax.random.PRNGKey(40))
    bais2  = jnp.zeros((output_dim, 1))

    # Training loop
    for epoch in range(epochs):
        dW1, dB1, dW2, dB2 = backprop(weight, bais, weight2, bais2, X.T, y_org, act_fun)
        update_WB(dW1, dB1, dW2, dB2, lr)

        if num_epo_print > 0 and ((epoch+1) % num_epo_print == 0 or epoch == epochs-1):
            hidden = forward_pass(weight, X.T, bais)
            hidden = act_fun(hidden)
            y_pred = forward_pass(weight2, hidden, bais2)
            loss = Loss_funtion(y_pred, y_org)
            print(f"Epoch {epoch+1} Training loss: {loss:.6f}")

    return weight, bais, weight2, bais2

# -----------------------------
# 5️⃣ Prediction function
# -----------------------------
def predict(X, weight1, bias1, weight2, bias2, act_fun=relu):
    hidden = forward_pass(weight1, X.T, bias1)
    hidden = act_fun(hidden)
    y_pred = forward_pass(weight2, hidden, bias2)
    return np.array(y_pred)

# -----------------------------
# 6️⃣ Load and preprocess training data
# -----------------------------
train_data = pd.read_csv('X_test.csv')
train_data = train_data.drop(columns=["CustomerId", "Surname"])
train_data["Gender"] = train_data["Gender"].map({"Female": 0, "Male": 1})
train_data["Gender"] = train_data["Gender"].fillna(train_data["Gender"].median())
train_data["Geography"] = train_data["Geography"].astype("category").cat.codes
train_data["Balance"] = train_data["Balance"] / 100000.0
train_data["EstimatedSalary"] = train_data["EstimatedSalary"] / 50000.0
train_data["CreditScore"] = train_data["CreditScore"] / 650

X = train_data.values.astype(np.float32)
y = pd.read_csv('y_test.csv').drop(columns=["CustomerId"]).values.astype(np.float32)

X_jax = jnp.array(X, dtype=jnp.float32)
y_jax = jnp.array(y, dtype=jnp.float32)

# -----------------------------
# 7️⃣ Train the network
# -----------------------------
weight, bais, weight2, bais2 = Personal_NN(
    X_jax, y_jax,
    lr=0.01,
    epochs=1000,
    act_fun=relu,
    num_epo_print=100
)

# -----------------------------
# 8️⃣ Load and preprocess test data
# -----------------------------
test_df = pd.read_csv('X_train.csv')
test_df = test_df.drop(columns=["CustomerId", "Surname"])
test_df["Gender"] = test_df["Gender"].map({"Female":0,"Male":1})
test_df["Geography"] = test_df["Geography"].astype("category").cat.codes
test_df["Balance"] = test_df["Balance"] / 100000.0
test_df["EstimatedSalary"] = test_df["EstimatedSalary"] / 50000.0
test_df["CreditScore"] = test_df["CreditScore"] / 650

X_test = test_df.values.astype(np.float32)
y_test = pd.read_csv('y_train.csv').drop(columns=["CustomerId"]).values.astype(np.float32)

X_test_jax = jnp.array(X_test, dtype=jnp.float32)
y_test_jax = jnp.array(y_test, dtype=jnp.float32).reshape(1, -1)

print("Shapes:", X_test_jax.shape, y_test_jax.shape)

# -----------------------------
# 9️⃣ Make predictions
# -----------------------------
y_pred_test = predict(X_test_jax, weight, bais, weight2, bais2)
y_pred_class = (y_pred_test > 0.5).astype(int)

# Add predictions to dataframe
test_df["Predicted_Exited"] = y_pred_class.T  # transpose to align

# Display first 5 predictions
display(test_df.head())


Epoch 100 Training loss: 0.169541
Epoch 200 Training loss: 0.162310
Epoch 300 Training loss: 0.162183
Epoch 400 Training loss: 0.162180
Epoch 500 Training loss: 0.162180
Epoch 600 Training loss: 0.162180
Epoch 700 Training loss: 0.162180
Epoch 800 Training loss: 0.162180
Epoch 900 Training loss: 0.162180
Epoch 1000 Training loss: 0.162180
Shapes: (6499, 10) (1, 6499)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Predicted_Exited
0,1.216923,1,0.0,35,7,0.524362,1,1,0,3.221035,0
1,1.084615,1,1.0,42,8,1.666859,2,1,1,1.106270,0
2,0.835385,0,0.0,31,4,1.383179,1,0,0,1.236875,0
3,1.090769,0,0.0,32,2,0.000000,2,0,0,2.193626,0
4,1.098462,1,0.0,36,1,1.016090,2,1,1,0.008955,0


In [99]:
print("Real value of Y test :- ")
print(y_test_jax[0, :5].astype(int)) # <- First 5 actual value

Real value of Y test :- 
[0 0 0 0 0]


## Personal ANN Training Accuracy

In [102]:
y_pred_class = (y_pred_test > 0.5).astype(int)
accuracy = np.mean(y_pred_class.T == y_test_jax)
print("Training accuracy:", accuracy)

Training accuracy: 0.79627633
